In [1]:
import os
os.chdir("..")                    # go up one level to project root
print("Now in:", os.getcwd())    # should show C:\Users\🍀\...
print(os.listdir("."))           # should show data, day1, day2 etc.

Now in: F:\finance-ai-course
['.claude.json', '.claude.json.backup', '.env', '.gitconfig', '.gitignore', '.lesshst', 'battery-report.html', 'data', 'day1', 'day10', 'day2', 'day3', 'day4', 'day5', 'day6', 'day7', 'day8', 'day9']


In [2]:
import pandas as pd

# Load the trial balance
df = pd.read_csv("data/data/trial_balance.csv")
print(df.head())
print(f"\nShape: {df.shape}")        # rows x columns
print(df.dtypes)                     # column data types

# --- Filter to one entity and one period ---
dec_a = df[(df["entity"] == "EntityA") & (df["period"] == "2025-12")]

# --- Build a P&L summary ---
income = dec_a[dec_a["account_type"] == "Income"]["credit"].sum()
cogs   = dec_a[dec_a["account_type"] == "Expense"]
cogs_total = cogs[cogs["account_code"] == 5001]["debit"].sum()
opex_total = cogs[cogs["account_code"] != 5001]["debit"].sum()

gross_profit = income - cogs_total
net_profit   = gross_profit - opex_total

print(f"\n=== EntityA P&L — Dec 2025 ===")
print(f"Revenue:        ${income:>12,.0f}")
print(f"COGS:          (${cogs_total:>11,.0f})")
print(f"Gross Profit:   ${gross_profit:>12,.0f}  ({gross_profit/income*100:.1f}%)")
print(f"OpEx:          (${opex_total:>11,.0f})")
print(f"Net Profit:     ${net_profit:>12,.0f}  ({net_profit/income*100:.1f}%)")

# --- Month-on-month variance ---
rev_dec = df[(df["entity"]=="EntityA") & (df["period"]=="2025-12") & (df["account_code"]==4001)]["credit"].sum()
rev_nov = df[(df["entity"]=="EntityA") & (df["period"]=="2025-11") & (df["account_code"]==4001)]["credit"].sum()
mom_var = rev_dec - rev_nov
print(f"\nRevenue MoM: ${mom_var:+,.0f} ({mom_var/rev_nov*100:+.1f}%)")

   account_code         account_name account_type     debit   credit   period  \
0          1001        Cash and Bank        Asset   45200.0      0.0  2025-12   
1          1100  Accounts Receivable        Asset  128400.0      0.0  2025-12   
2          1200            Inventory        Asset   67800.0      0.0  2025-12   
3          1500         Fixed Assets        Asset  320000.0      0.0  2025-12   
4          2001     Accounts Payable    Liability       0.0  89500.0  2025-12   

    entity  
0  EntityA  
1  EntityA  
2  EntityA  
3  EntityA  
4  EntityA  

Shape: (23, 7)
account_code      int64
account_name        str
account_type        str
debit           float64
credit          float64
period              str
entity              str
dtype: object

=== EntityA P&L — Dec 2025 ===
Revenue:        $     462,000
COGS:          ($    220,000)
Gross Profit:   $     242,000  (52.4%)
OpEx:          ($    118,600)
Net Profit:     $     123,400  (26.7%)

Revenue MoM: $+25,000 (+6.5%)


In [3]:
import os
if os.path.basename(os.getcwd()).startswith("day"):
    os.chdir("..")
print("Working from:", os.getcwd())

Working from: F:\finance-ai-course


In [7]:
import pandas as pd

def analyse_trial_balance(filepath: str, entity: str, current_period: str, prior_period: str = None):
    """
    Accepts any trial balance CSV and outputs:
    - P&L summary
    - Gross and net margins
    - Month-on-month revenue variance (if prior period provided)
    """
    # --- Load ---
    df = pd.read_csv(filepath)

    # --- Filter to entity and period ---
    curr = df[(df["entity"] == entity) & (df["period"] == current_period)]

    if curr.empty:
        print(f"No data found for {entity} / {current_period}")
        return

    # --- Compute P&L lines ---
    revenue  = curr[curr["account_type"] == "Income"]["credit"].sum()
    cogs     = curr[curr["account_code"].astype(str) == "5001"]["debit"].sum()
    opex     = curr[
                   (curr["account_type"] == "Expense") &
                   (curr["account_code"].astype(str) != "5001")
               ]["debit"].sum()

    gross    = revenue - cogs
    net      = gross - opex

    gm_pct   = round(gross / revenue * 100, 1) if revenue else 0
    nm_pct   = round(net   / revenue * 100, 1) if revenue else 0

    # --- Print P&L summary ---
    print("=" * 45)
    print(f"  P&L SUMMARY — {entity}  |  {current_period}")
    print("=" * 45)
    print(f"  Revenue          ${revenue:>12,.0f}")
    print(f"  COGS            (${cogs:>11,.0f})")
    print(f"  {'─'*38}")
    print(f"  Gross Profit     ${gross:>12,.0f}  ({gm_pct}%)")
    print(f"  OpEx            (${opex:>11,.0f})")
    print(f"  {'─'*38}")
    print(f"  Net Profit       ${net:>12,.0f}  ({nm_pct}%)")
    print("=" * 45)

    # --- Month-on-month variance (optional) ---
    if prior_period:
        prev = df[(df["entity"] == entity) & (df["period"] == prior_period)]
        prev_rev = prev[prev["account_type"] == "Income"]["credit"].sum()

        if prev_rev:
            mom_var     = revenue - prev_rev
            mom_pct     = round(mom_var / prev_rev * 100, 1)
            direction   = "▲" if mom_var >= 0 else "▼"
            print(f"\n  Revenue vs {prior_period}:")
            print(f"  {direction} ${mom_var:+,.0f}  ({mom_pct:+.1f}% MoM)")
            print("=" * 45)

    return {
        "entity": entity,
        "period": current_period,
        "revenue": revenue,
        "gross_profit": gross,
        "gross_margin_pct": gm_pct,
        "net_profit": net,
        "net_margin_pct": nm_pct,
    }


# --- Run it ---
result = analyse_trial_balance(
    filepath       = "data/data/trial_balance.csv",
    entity         = "EntityA",
    current_period = "2025-12",
    prior_period   = "2025-11"
)

  P&L SUMMARY — EntityA  |  2025-12
  Revenue          $     462,000
  COGS            ($    220,000)
  ──────────────────────────────────────
  Gross Profit     $     242,000  (52.4%)
  OpEx            ($    118,600)
  ──────────────────────────────────────
  Net Profit       $     123,400  (26.7%)

  Revenue vs 2025-11:
  ▲ $+77,000  (+20.0% MoM)


In [8]:
script = '''import os, pandas as pd

def analyse_trial_balance(filepath, entity, current_period, prior_period=None):
    df = pd.read_csv(filepath)
    curr = df[(df["entity"] == entity) & (df["period"] == current_period)]
    if curr.empty:
        print(f"No data found for {entity} / {current_period}")
        return
    revenue = curr[curr["account_type"] == "Income"]["credit"].sum()
    cogs    = curr[curr["account_code"].astype(str) == "5001"]["debit"].sum()
    opex    = curr[(curr["account_type"] == "Expense") & (curr["account_code"].astype(str) != "5001")]["debit"].sum()
    gross   = revenue - cogs
    net     = gross - opex
    gm_pct  = round(gross / revenue * 100, 1) if revenue else 0
    nm_pct  = round(net   / revenue * 100, 1) if revenue else 0
    print("=" * 45)
    print(f"  P&L SUMMARY — {entity}  |  {current_period}")
    print("=" * 45)
    print(f"  Revenue          ${revenue:>12,.0f}")
    print(f"  COGS            (${cogs:>11,.0f})")
    print(f"  Gross Profit     ${gross:>12,.0f}  ({gm_pct}%)")
    print(f"  OpEx            (${opex:>11,.0f})")
    print(f"  Net Profit       ${net:>12,.0f}  ({nm_pct}%)")
    print("=" * 45)
    if prior_period:
        prev_rev = df[(df["entity"]==entity)&(df["period"]==prior_period)]
        prev_rev = prev_rev[prev_rev["account_type"]=="Income"]["credit"].sum()
        if prev_rev:
            mom = revenue - prev_rev
            print(f"  Revenue vs {prior_period}: ${mom:+,.0f} ({mom/prev_rev*100:+.1f}% MoM)")
            print("=" * 45)

if __name__ == "__main__":
    os.chdir(r"F:\\finance-ai-course")
    analyse_trial_balance("data/trial_balance.csv", "EntityA", "2025-12", "2025-11")
    analyse_trial_balance("data/trial_balance.csv", "EntityB", "2025-12")
'''

with open("day1/tb_analyser.py", "w") as f:
    f.write(script)

print("Saved to day1/tb_analyser.py")

Saved to day1/tb_analyser.py
